In [ ]:
# ==============================================================================
# 1. MOUNT DRIVE & EXTRACT ALL ARCHIVES
# ==============================================================================
import os
import glob
import subprocess
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupShuffleSplit
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/nih_chest_xray"
EXTRACT_DIR = "/content/nih_chest_xray_images"
CSV_PATH = os.path.join(DRIVE_DIR, "nih-cxr-lt_single-label_train.csv")

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Find and extract all .tar archives present in Drive
tar_files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tar")))
print(f"Found {len(tar_files)} TAR file(s). Extracting...")

for tar_path in tar_files:
    print(f"Extracting {os.path.basename(tar_path)}...")
    subprocess.run(["tar", "-xf", tar_path, "-C", EXTRACT_DIR], check=True)

print("Extraction completed successfully.")

Mounted at /content/drive
Found 5 TAR file(s). Extracting...
Extracting images_001.tar...
Extracting images_002.tar...
Extracting images_003.tar...
Extracting images_004.tar...
Extracting images_005.tar...
Extraction completed successfully.


In [ ]:
# ==============================================================================
# 2. METADATA MATCHING & PATIENT-AWARE DATA SPLITTING
# ==============================================================================
df = pd.read_csv(CSV_PATH)

# Map image filepaths from extracted disk contents
all_images = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.png"), recursive=True)
file_lookup = {os.path.basename(f): f for f in all_images}

df["filepath"] = df["id"].map(file_lookup)
df = df.dropna(subset=["filepath"]).reset_index(drop=True)
print(f"Total matched samples available: {len(df)}")

label_cols = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
    'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass',
    'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax',
    'Pneumoperitoneum', 'Pneumomediastinum', 'Subcutaneous Emphysema',
    'Tortuous Aorta', 'Calcification of the Aorta', 'No Finding'
]
NUM_CLASSES = len(label_cols)

# Patient-level split to prevent data leakage (70% train, 15% val, 15% test)
gss_temp = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(gss_temp.split(df, groups=df["subject_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df["subject_id"]))

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Unique patients: {train_df['subject_id'].nunique()}")
print(f"Val samples:   {len(val_df)} | Unique patients: {val_df['subject_id'].nunique()}")
print(f"Test samples:  {len(test_df)} | Unique patients: {test_df['subject_id'].nunique()}")

Total matched samples available: 28461
Train samples: 19988 | Unique patients: 6188
Val samples:   3917 | Unique patients: 1326
Test samples:  4556 | Unique patients: 1326


In [ ]:
# ==============================================================================
# 3. TF.DATA PIPELINE WITH MEDICAL IMAGE AUGMENTATION
# ==============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomZoom(height_factor=0.08, width_factor=0.08)
])

def create_dataset(dataframe, shuffle=False, augment=False):
    paths = dataframe["filepath"].values
    labels = dataframe[label_cols].values.astype("float32")

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    def parse_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_png(image, channels=1)
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
        image = tf.cast(image, tf.float32) / 255.0
        # Convert 1-channel grayscale to 3-channel RGB for DenseNet
        image = tf.image.grayscale_to_rgb(image)
        return image, label

    dataset = dataset.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=2048, seed=42)

    dataset = dataset.batch(BATCH_SIZE)

    if augment:
        dataset = dataset.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    return dataset.prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_df, shuffle=True, augment=True)
val_ds = create_dataset(val_df, shuffle=False, augment=False)
test_ds = create_dataset(test_df, shuffle=False, augment=False)

In [ ]:
# ==============================================================================
# 4. MODEL ARCHITECTURE (DENSENET121 BACKBONE)
# ==============================================================================
from tensorflow.keras.applications import DenseNet121

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='sigmoid')(x)

model = models.Model(inputs, outputs, name="ChestXRay_DenseNet121")
model.summary()

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "ChestXRay_DenseNet121"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │         5,140 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,306,068 (27.87 MB)

 Trainable params: 268,052 (1.02 MB)

 Non-trainable params: 7,038,016 (26.85 MB)

In [ ]:
# ==============================================================================
# 5. PHASE 1: WARMUP TRAINING (FROZEN BACKBONE)
# ==============================================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryFocalCrossentropy(
        apply_class_balancing=True,
        alpha=0.25,
        gamma=2.0
    ),
    metrics=[
        tf.keras.metrics.AUC(multi_label=True, name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='binary_accuracy')
    ]
)

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        mode='max',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_phase1_model.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

print("--- Starting Phase 1 Training (Feature Extraction) ---")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_phase1
)

--- Starting Phase 1 Training (Feature Extraction) ---
Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 863ms/step - auc: 0.5048 - binary_accuracy: 0.8033 - loss: 0.1027
Epoch 1: val_auc improved from None to 0.62059, saving model to best_phase1_model.keras

Epoch 1: finished saving model to best_phase1_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 697s 1s/step - auc: 0.5372 - binary_accuracy: 0.9084 - loss: 0.0406 - val_auc: 0.6206 - val_binary_accuracy: 0.9662 - val_loss: 0.0102 - learning_rate: 0.0010
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 823ms/step - auc: 0.5579 - binary_accuracy: 0.9597 - loss: 0.0114
Epoch 2: val_auc improved from 0.62059 to 0.66058, saving model to best_phase1_model.keras

Epoch 2: finished saving model to best_phase1_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 600s 914ms/step - auc: 0.5861 - binary_accuracy: 0.9599 - loss: 0.0111 - val_auc: 0.6606 - val_binary_accuracy: 0.9650 - val_loss: 0.0101 - learning_rate: 0.0010
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 809ms/st